# Train a YOLO Model

### Detailed Code Review

#### 1. `CustomAssigner`

This is the heart of your implementation, and it's very well done.

*   **Logic:** The flow is perfect.
    1.  Perform the standard assignment to get an initial `fg_mask` (foreground) and `bg_mask` (background).
    2.  Isolate the model's scores (`pd_scores`) for only the background regions.
    3.  Find the maximum class score for each background anchor. This is a clever proxy for "objectness" or the model's confidence that *something* is there.
    4.  Filter these scores by `correction_thresh` to identify the anchors that are likely unlabeled positives.
    5.  Modify the `target_scores` tensor by setting the value for these specific anchors to `-1.0`.

*   **Implementation:** The use of `torch.where` and advanced indexing (`pd_scores[bg_indices]`) is efficient and vectorized. There are no loops, which is exactly what you want for performance.

#### 2. `CustomV8DetectionLoss`

This class serves as the perfect bridge between your new assigner and the rest of the YOLOv8 loss calculation.

*   **Initialization:** You correctly retrieve the `correction_thresh` from the model's arguments (`model.args`). This is the standard ultralytics pattern and makes your feature configurable from the command line or `train()` call. Replacing `self.assigner` is exactly the right thing to do.
*   **Loss Calculation (`__call__`)**:
    *   The most critical part is `cls_mask = (target_scores != -1.0)`. This is the payoff for the work done in the assigner. It creates a boolean mask that is `True` for all normal positive and negative samples but `False` for your "corrected" negative samples.
    *   Applying this mask `pred_scores[cls_mask]` and `target_scores[cls_mask]` before passing them to the BCE loss function elegantly excludes the unwanted samples from the classification loss.
    *   **Subtle but Important Point:** The classification loss is normalized by `num_pos` (the number of foreground/positive objects). By removing some negative samples from the loss calculation (`loss_cls_unnormalized.sum()` will be smaller), but keeping the normalizer the same, you are effectively reducing the penalty from the classification component of the total loss. This is the desired behavior—it stops the model from being overly punished for finding new objects.

#### 3. `CustomTrainer`

This is the final piece of the puzzle, and it's implemented perfectly.

*   **Injection:** Overriding `_setup_train` is the correct, non-intrusive way to inject your custom components.
*   **Execution Order:** Calling `super()._setup_train()` first ensures all standard setup is complete before you overwrite the `criterion`.
*   **Device Management:** The `self.model.criterion.assigner.to(self.device)` line is crucial and often forgotten. Since `CustomAssigner` is an `nn.Module`, its parameters and buffers must be on the same device as the model's data. You've handled this correctly.

---

### Potential Considerations & Improvements

The code is already excellent, but here are some things to think about during practical application:

1.  **Hyperparameter Sensitivity (`correction_thresh`):** This is now the most critical hyperparameter in your training process.
    *   If `correction_thresh` is **too low** (e.g., 0.1), you might start ignoring *true negative* samples that the model is slightly unsure about. This could lead to an increase in false positives, as the model isn't penalized for making low-confidence mistakes in the background.
    *   If `correction_thresh` is **too high** (e.g., 0.8), the feature may have little to no effect, as only extremely confident predictions will be ignored.
    *   **Recommendation:** You will need to tune this value on a validation set. A good starting point might be somewhere in the `0.3` to `0.5` range.

2.  **Early Training Instability:** In the first few epochs, the model's predictions are essentially random. A randomly initialized model might produce high-confidence scores by chance. Your logic would then trust these noisy predictions and ignore those background samples, preventing the model from learning that they are, in fact, background.
    *   **Potential Improvement (Curriculum Learning):** You could implement a schedule for `correction_thresh`. Start with `correction_thresh = 1.0` (disabling the feature) for the first few epochs, and then gradually anneal it down to your target value (e.g., 0.3) as the model becomes more stable and its predictions more reliable.

In [1]:
import torch
import torch.nn as nn
from ultralytics.utils.tal import TaskAlignedAssigner, make_anchors
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.utils.loss import v8DetectionLoss
from ultralytics.utils import LOGGER, RANK
from ultralytics.utils.metrics import bbox_iou
from typing import Any, Dict, Tuple

# ---------------------------------------------------------------------------
# 1. Custom Assigner (with Negative Label Correction)
# ---------------------------------------------------------------------------

class CustomAssigner(nn.Module):
    """
    A custom task-aligned assigner that implements "Negative Label Correction".
    It drops background samples where the model predicts an object with high confidence,
    assuming these are unlabeled positives from a partially-labeled dataset.
    This prevents the model from being penalized for finding new, correct objects.
    """

    def __init__(self, topk: int = 10, num_classes: int = 80, alpha: float = 0.5, beta: float = 6.0, eps: float = 1e-9, 
                 correction_thresh: float = 0.3):
        """
        Initialize the CustomAssigner.
        
        Args:
            topk (int): The number of top candidates to consider for assignment.
            num_classes (int): The number of object classes.
            alpha (float): The alpha parameter for the alignment metric.
            beta (float): The beta parameter for the alignment metric.
            eps (float): A small value to prevent division by zero.
            correction_thresh (float): Confidence threshold. Background samples with a max class score
                                     ABOVE this value will be dropped from the loss calculation.
                                     A value of 0.0 disables this feature.
        """
        super().__init__()
        self.topk = topk
        self.num_classes = num_classes
        self.alpha = alpha
        self.beta = beta
        self.eps = eps
        
        # --- Our custom parameter for Negative Label Correction ---
        self.correction_thresh = correction_thresh
        
        if RANK in (-1, 0):
            print(f"✅ CustomAssigner initialized.")
            if self.correction_thresh > 0.0:
                print(f"   - Negative Label Correction: ENABLED (threshold = {self.correction_thresh})")
            else:
                print(f"   - Negative Label Correction: DISABLED")

    @torch.no_grad()
    def forward(self, pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt):
        """
        Computes the assignment and then applies our Negative Label Correction logic.
        """
        self.bs = pd_scores.shape[0]
        self.n_max_boxes = gt_bboxes.shape[1]
        device = gt_bboxes.device

        if self.n_max_boxes == 0:
            return (
                torch.full_like(pd_scores[..., 0], self.num_classes),
                torch.zeros_like(pd_bboxes),
                torch.zeros_like(pd_scores),
                torch.zeros_like(pd_scores[..., 0]),
                torch.zeros_like(pd_scores[..., 0]),
            )

        try:
            # Perform the standard assignment logic by calling the internal _forward method
            results = self._forward(pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt)
        except torch.cuda.OutOfMemoryError:
            LOGGER.warning("CUDA OutOfMemoryError in CustomAssigner, using CPU")
            cpu_tensors = [t.cpu() for t in (pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt)]
            results = self._forward(*cpu_tensors)
            results = tuple(t.to(device) for t in results)

        target_labels, target_bboxes, target_scores, fg_mask, target_gt_idx = results

        if self.training and self.correction_thresh > 0.0:
            # --- NEGATIVE LABEL CORRECTION LOGIC ---
            
            # 1. Identify all initial background regions
            bg_mask = ~fg_mask
            bg_indices = torch.where(bg_mask)

            if bg_indices[0].numel() > 0:
                # 2. Get the model's multi-class prediction scores for these background regions
                bg_scores_all_classes = pd_scores[bg_indices]
                
                # 3. Find the max score for each background anchor (its "objectness" confidence)
                bg_max_scores, _ = bg_scores_all_classes.max(dim=-1)

                # 4. Identify which samples have a confidence ABOVE our correction threshold.
                # These are the samples we suspect are unlabeled positives.
                mask_to_drop = bg_max_scores > self.correction_thresh
                
                # 5. Get the original indices of the samples to drop.
                batch_idx_to_drop = bg_indices[0][mask_to_drop]
                anchor_idx_to_drop = bg_indices[1][mask_to_drop]

                # 6. Exclude these suspected unlabeled positives from the loss calculation
                # by setting their target score to -1.
                target_scores[batch_idx_to_drop, anchor_idx_to_drop] = -1.0

        return target_labels, target_bboxes, target_scores, fg_mask, target_gt_idx

    # --- ALL HELPER METHODS ARE COPIED DIRECTLY FROM THE ORIGINAL ULTRALYTICS IMPLEMENTATION ---

    def _forward(self, pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt):
        """Internal forward pass for assignment."""
        mask_pos, align_metric, overlaps = self.get_pos_mask(
            pd_scores, pd_bboxes, gt_labels, gt_bboxes, anc_points, mask_gt
        )
        target_gt_idx, fg_mask, mask_pos = self.select_highest_overlaps(mask_pos, overlaps, self.n_max_boxes)
        target_labels, target_bboxes, target_scores = self.get_targets(gt_labels, gt_bboxes, target_gt_idx, fg_mask)
        align_metric *= mask_pos
        pos_align_metrics = align_metric.amax(dim=-1, keepdim=True)
        pos_overlaps = (overlaps * mask_pos).amax(dim=-1, keepdim=True)
        norm_align_metric = (align_metric * pos_overlaps / (pos_align_metrics + self.eps)).amax(-2).unsqueeze(-1)
        target_scores = target_scores * norm_align_metric
        return target_labels, target_bboxes, target_scores, fg_mask.bool(), target_gt_idx

    def get_pos_mask(self, pd_scores, pd_bboxes, gt_labels, gt_bboxes, anc_points, mask_gt):
        """Get positive mask."""
        mask_in_gts = self.select_candidates_in_gts(anc_points, gt_bboxes)
        align_metric, overlaps = self.get_box_metrics(pd_scores, pd_bboxes, gt_labels, gt_bboxes, mask_in_gts * mask_gt)
        mask_topk = self.select_topk_candidates(align_metric, topk_mask=mask_gt.expand(-1, -1, self.topk).bool())
        mask_pos = mask_topk * mask_in_gts * mask_gt
        return mask_pos, align_metric, overlaps

    def get_box_metrics(self, pd_scores, pd_bboxes, gt_labels, gt_bboxes, mask_gt):
        """Compute alignment metric."""
        na = pd_bboxes.shape[-2]
        mask_gt = mask_gt.bool()
        overlaps = torch.zeros([self.bs, self.n_max_boxes, na], dtype=pd_bboxes.dtype, device=pd_bboxes.device)
        bbox_scores = torch.zeros([self.bs, self.n_max_boxes, na], dtype=pd_scores.dtype, device=pd_scores.device)
        ind = torch.zeros([2, self.bs, self.n_max_boxes], dtype=torch.long)
        ind[0] = torch.arange(end=self.bs).view(-1, 1).expand(-1, self.n_max_boxes)
        ind[1] = gt_labels.squeeze(-1)
        bbox_scores[mask_gt] = pd_scores[ind[0], :, ind[1]][mask_gt]
        pd_boxes = pd_bboxes.unsqueeze(1).expand(-1, self.n_max_boxes, -1, -1)[mask_gt]
        gt_boxes = gt_bboxes.unsqueeze(2).expand(-1, -1, na, -1)[mask_gt]
        overlaps[mask_gt] = self.iou_calculation(gt_boxes, pd_boxes)
        align_metric = bbox_scores.pow(self.alpha) * overlaps.pow(self.beta)
        return align_metric, overlaps

    def iou_calculation(self, gt_bboxes, pd_bboxes):
        """Calculate IoU."""
        return bbox_iou(gt_bboxes, pd_bboxes, xywh=False, CIoU=True).squeeze(-1).clamp_(0)

    def select_topk_candidates(self, metrics, topk_mask=None):
        """Select top-k candidates."""
        topk_metrics, topk_idxs = torch.topk(metrics, self.topk, dim=-1, largest=True)
        if topk_mask is None:
            topk_mask = (topk_metrics.max(-1, keepdim=True)[0] > self.eps).expand_as(topk_idxs)
        topk_idxs.masked_fill_(~topk_mask, 0)
        count_tensor = torch.zeros(metrics.shape, dtype=torch.int8, device=topk_idxs.device)
        ones = torch.ones_like(topk_idxs[:, :, :1], dtype=torch.int8, device=topk_idxs.device)
        for k in range(self.topk):
            count_tensor.scatter_add_(-1, topk_idxs[:, :, k : k + 1], ones)
        count_tensor.masked_fill_(count_tensor > 1, 0)
        return count_tensor.to(metrics.dtype)

    def get_targets(self, gt_labels, gt_bboxes, target_gt_idx, fg_mask):
        """Compute target labels, bboxes and scores."""
        batch_ind = torch.arange(end=self.bs, dtype=torch.int64, device=gt_labels.device)[..., None]
        target_gt_idx = target_gt_idx + batch_ind * self.n_max_boxes
        target_labels = gt_labels.long().flatten()[target_gt_idx]
        target_bboxes = gt_bboxes.view(-1, gt_bboxes.shape[-1])[target_gt_idx]
        target_labels.clamp_(0)
        target_scores = torch.zeros((target_labels.shape[0], target_labels.shape[1], self.num_classes),
                                    dtype=torch.int64, device=target_labels.device)
        target_scores.scatter_(2, target_labels.unsqueeze(-1), 1)
        fg_scores_mask = fg_mask[:, :, None].repeat(1, 1, self.num_classes)
        target_scores = torch.where(fg_scores_mask > 0, target_scores, 0)
        return target_labels, target_bboxes, target_scores

    @staticmethod
    def select_candidates_in_gts(xy_centers, gt_bboxes, eps=1e-9):
        """Select anchors in ground truth boxes."""
        n_anchors = xy_centers.shape[0]
        bs, n_boxes, _ = gt_bboxes.shape
        lt, rb = gt_bboxes.view(-1, 1, 4).chunk(2, 2)
        bbox_deltas = torch.cat((xy_centers[None] - lt, rb - xy_centers[None]), dim=2).view(bs, n_boxes, n_anchors, -1)
        return bbox_deltas.amin(3).gt_(eps)

    @staticmethod
    def select_highest_overlaps(mask_pos, overlaps, n_max_boxes):
        """Handle multiple gt assignments."""
        fg_mask = mask_pos.sum(-2)
        if fg_mask.max() > 1:
            mask_multi_gts = (fg_mask.unsqueeze(1) > 1).expand(-1, n_max_boxes, -1)
            max_overlaps_idx = overlaps.argmax(1)
            is_max_overlaps = torch.zeros(mask_pos.shape, dtype=mask_pos.dtype, device=mask_pos.device)
            is_max_overlaps.scatter_(1, max_overlaps_idx.unsqueeze(1), 1)
            mask_pos = torch.where(mask_multi_gts, is_max_overlaps, mask_pos).float()
            fg_mask = mask_pos.sum(-2)
        target_gt_idx = mask_pos.argmax(-2)
        return target_gt_idx, fg_mask, mask_pos


# ---------------------------------------------------------------------------
# 2. Custom Loss Class (A clean way to package the change)
# ---------------------------------------------------------------------------

class CustomV8DetectionLoss(v8DetectionLoss):
    """
    A custom version of v8DetectionLoss that uses our CustomAssigner
    to implement Negative Label Correction for partially-labeled datasets.
    """
    def __init__(self, model):
        super().__init__(model)
        if not hasattr(self, 'bbox_decode'):
            self.bbox_decode = self.decode_bboxes
        
        # Get the custom argument from the training configuration (e.g., from trainer.train(correction_thresh=0.3))
        correction_thresh = getattr(model.args, 'correction_thresh', 0.30)  # <----------------------------
        
        # Replace the default assigner with our custom one
        self.assigner = CustomAssigner(topk=10, 
                                       num_classes=self.nc, 
                                       alpha=0.5, 
                                       beta=6.0, 
                                       correction_thresh=correction_thresh)
        if RANK in (-1, 0):
            print("✅ CustomV8DetectionLoss initialized and assigner replaced.")

    def __call__(self, preds, batch):
        """
        Calculates the loss using the custom assigner.
        The -1 target scores from the assigner are automatically handled by the cls_mask.
        """
        loss = torch.zeros(3, device=self.device)  # box, cls, dfl
        feats = preds[1] if isinstance(preds, tuple) else preds
        pred_distri, pred_scores = torch.cat([xi.view(feats[0].shape[0], self.no, -1) for xi in feats], 2).split(
            (self.reg_max * 4, self.nc), 1)

        pred_scores = pred_scores.permute(0, 2, 1).contiguous()
        pred_distri = pred_distri.permute(0, 2, 1).contiguous()

        dtype = pred_scores.dtype
        batch_size = pred_scores.shape[0]
        imgsz = torch.tensor(feats[0].shape[2:], device=self.device, dtype=dtype) * self.stride[0]
        anchor_points, stride_tensor = make_anchors(feats, self.stride, 0.5)

        targets = torch.cat((batch['batch_idx'].view(-1, 1), batch['cls'].view(-1, 1), batch['bboxes']), 1)
        targets = self.preprocess(targets.to(self.device), batch_size, scale_tensor=imgsz[[1, 0, 1, 0]])
        gt_labels, gt_bboxes = targets.split((1, 4), 2)
        mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0)

        pred_bboxes = self.bbox_decode(anchor_points, pred_distri)

        _, target_bboxes, target_scores, fg_mask, _ = self.assigner(
            pred_scores.detach().sigmoid(),
            (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype),
            anchor_points * stride_tensor,
            gt_labels,
            gt_bboxes,
            mask_gt)
        
        target_scores_sum = max(target_scores[target_scores > 0].sum(), 1)

        # Classification loss: Create a mask to select only targets that are NOT -1.0
        # This correctly handles positives, true negatives, and ignores corrected negatives.
        cls_mask = (target_scores != -1.0)
        
        loss_cls_unnormalized = self.bce(pred_scores[cls_mask], target_scores[cls_mask].to(dtype))

        num_pos = fg_mask.sum()
        if num_pos > 0:
            loss[1] = loss_cls_unnormalized.sum() / num_pos
        
        # Bbox loss (this part is correct and unaffected)
        if fg_mask.sum():
            target_bboxes /= stride_tensor
            try:
                loss[0], loss[2] = self.bbox_loss(
                    pred_distri, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum, fg_mask)
            except TypeError:
                loss[0], loss[2] = self.bbox_loss(
                    pred_distri, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum)

        loss[0] *= self.hyp.box
        loss[1] *= self.hyp.cls
        loss[2] *= self.hyp.dfl

        return loss.sum() * batch_size, loss.detach()


# ---------------------------------------------------------------------------
# 3. Custom Trainer (Injects the custom loss function)
# ---------------------------------------------------------------------------

class CustomTrainer(DetectionTrainer):
    """
    This trainer injects our custom loss function and adds a dynamic, ADAPTIVE
    scheduling mechanism for the `correction_thresh` parameter.

    The scheduler has two modes, set via the `thresh_schedule` argument:
    1. 'static' (default): Uses a fixed `correction_thresh` throughout training.
    2. 'adaptive': Dynamically sets the threshold each epoch based on the
       confidence scores of false positives from the validation set.
    """

    def _setup_train(self, world_size):
        """
        Overrides the training setup to replace the model's loss function
        and initialize the threshold scheduler.
        """
        super()._setup_train(world_size)

        # 1. Replace the standard loss with our custom version.
        # This will read the initial `correction_thresh` from the args.
        if RANK in (-1, 0):
            LOGGER.info("✅ Replacing the model's criterion with CustomV8DetectionLoss.")
        self.model.criterion = CustomV8DetectionLoss(self.model)
        self.model.criterion.assigner.to(self.device)

        # 2. Setup the threshold scheduler.
        self._setup_threshold_scheduler()
        
        # 3. Register the custom callback to update the threshold at the end of each epoch.
        self.add_callback("on_fit_epoch_end", self._update_threshold)
        if RANK in (-1, 0):
            print("✅ Registered custom callback for end-of-epoch statistics.")

    def _setup_threshold_scheduler(self):
        """
        Initializes parameters for the threshold scheduler based on training args.
        """
        self.thresh_schedule = "adaptive"  # "static" or "adaptive"

        if self.thresh_schedule == 'adaptive':
            # --- Parameters for adaptive mode ---
            self.adaptive_percentile =  0.99
            self.adaptive_start_epoch =  3
            
            # Use `correction_thresh` and `end_thresh` as the min/max clamp values
            self.min_thresh =  0.05
            self.max_thresh =  0.20

            if RANK in (-1, 0):
                LOGGER.info("📈 Initializing 'adaptive' scheduler for correction_thresh:")
                LOGGER.info(f"   - Mode: Set to the {self.adaptive_percentile*100:.0f}th percentile of FP confidence.")
                LOGGER.info(f"   - Warm-up: Adaptive updates begin at epoch {self.adaptive_start_epoch}.")
                LOGGER.info(f"   - Clamp Range: Threshold will be kept between [{self.min_thresh}, {self.max_thresh}].")
        else:
            # This covers the 'static' case
            if RANK in (-1, 0):
                initial_thresh = getattr(self.args, 'correction_thresh', 0.0)
                LOGGER.info(f"   - Using a static correction_thresh of {initial_thresh} (no scheduling).")

    def _update_threshold(self, *args, **kwargs):
        """
        Called at the end of each epoch (after validation).
        Updates the threshold if in 'adaptive' mode and logs it.
        """
        # This is the primary method to modify.
        
        if self.thresh_schedule == 'adaptive':
            # Sanity check that the custom assigner is in place
            if not (hasattr(self.model.criterion, 'assigner') and
                    isinstance(self.model.criterion.assigner, CustomAssigner)):
                return

            current_epoch = self.epoch + 1
            # Get the current threshold to modify it
            new_thresh = self.model.criterion.assigner.correction_thresh

            if current_epoch < self.adaptive_start_epoch:
                # During warm-up, set the threshold to the minimum value
                new_thresh = self.min_thresh
                if RANK in (-1, 0) and current_epoch == self.adaptive_start_epoch - 1:
                    LOGGER.info(f"Warm-up complete. Starting adaptive threshold updates next epoch.")
            else:
                validator = self.validator
                if validator and hasattr(validator, 'stats') and validator.stats:
                    stats = validator.stats
                    conf_scores = stats.get('conf')
                    tp_mask = stats.get('tp')

                    # Handle list concatenation from validator if using DDP
                    if isinstance(conf_scores, list):
                        conf_scores = torch.cat(conf_scores) if conf_scores else torch.tensor([])
                    if isinstance(tp_mask, list):
                        tp_mask = torch.cat(tp_mask) if tp_mask else torch.tensor([])

                    if conf_scores is not None and tp_mask is not None and conf_scores.numel() > 0:
                        
                        # --- ✅ THE KEY FIX ---
                        # If tp_mask is 2D (detections, classes), reduce it to a 1D mask.
                        # A detection is a True Positive if it's a TP for *any* class.
                        if tp_mask.ndim == 2:
                            tp_mask_1d = torch.any(tp_mask, dim=1)
                        else: # It's already 1D
                            tp_mask_1d = tp_mask

                        # Ensure shapes are compatible before indexing
                        if conf_scores.shape[0] == tp_mask_1d.shape[0]:
                            is_fp = ~tp_mask_1d.bool()
                            fp_scores = conf_scores[is_fp]

                            if fp_scores.numel() > 0:
                                if RANK in (-1, 0):
                                    LOGGER.info(f"Calculating new threshold based on {fp_scores.numel()} False Positives.")
                                # Calculate the new threshold from the confidence of FPs
                                calculated_thresh = torch.quantile(fp_scores, self.adaptive_percentile)
                                # Clamp the value within the defined min/max range
                                new_thresh = torch.clamp(calculated_thresh, self.min_thresh, self.max_thresh).item()
                            else:
                                if RANK in (-1, 0):
                                    LOGGER.info("No False Positives found in validation set; threshold not updated.")
                        else:
                            if RANK in (-1, 0):
                                LOGGER.warning(f"Shape mismatch between conf_scores ({conf_scores.shape}) and tp_mask ({tp_mask_1d.shape}). Threshold not updated.")
                    else:
                        if RANK in (-1, 0):
                            LOGGER.warning("Validator stats missing data or empty; threshold not updated.")
                else:
                    if RANK in (-1, 0):
                        LOGGER.warning("Validator results not available; threshold not updated this epoch.")

            # Update the threshold in the assigner for the next training epoch
            if self.model.criterion.assigner.correction_thresh != new_thresh:
                self.model.criterion.assigner.correction_thresh = new_thresh
                if RANK in (-1, 0):
                    LOGGER.info(f"🔄 Updated correction_thresh to {new_thresh:.4f} for epoch {current_epoch + 1}.")

    thresh_schedule='adaptive',
    
    # Set the floor slightly below the target, e.g., 0.40
    # This gives the model room to be less confident early on.
    correction_thresh=0.40,
    
    # Set the ceiling slightly above the target, e.g., 0.75
    # This allows the model to become more confident than the static baseline.
    end_thresh=0.75,
    
    # Keep percentile and warm-up at sensible defaults
    adaptive_percentile=0.99,
    adaptive_start_epoch=10,

In [2]:
import numpy as np
import ultralytics.data.build as detection_build
from ultralytics.data.dataset import YOLODataset

class WeightedInstanceDataset(YOLODataset):
    def __init__(self, *args, mode="train", **kwargs):
        """
        Initialize the WeightedDataset.

        Args:
            class_weights (list or numpy array): A list or array of weights corresponding to each class.
        """
        super(WeightedInstanceDataset, self).__init__(*args, **kwargs)

        self.train_mode = "train" in self.prefix

        # You can also specify weights manually instead
        self.count_instances()
        class_weights = np.sum(self.counts) / self.counts

        # Aggregation function
        self.agg_func = np.mean

        self.class_weights = np.array(class_weights)
        self.weights = self.calculate_weights()
        self.probabilities = self.calculate_probabilities()

    def count_instances(self):
        """
        Count the number of instances per class

        Returns:
            dict: A dict containing the counts for each class.
        """
        self.counts = [0 for i in range(len(self.data["names"]))]
        for label in self.labels:
            cls = label['cls'].reshape(-1).astype(int)
            for id in cls:
                self.counts[id] += 1

        self.counts = np.array(self.counts)
        self.counts = np.where(self.counts == 0, 1, self.counts)

    def calculate_weights(self):
        """
        Calculate the aggregated weight for each label based on class weights.

        Returns:
            list: A list of aggregated weights corresponding to each label.
        """
        weights = []
        for label in self.labels:
            cls = label['cls'].reshape(-1).astype(int)

            # Give a default weight to background class
            if cls.size == 0:
                weights.append(1)
                continue

            # Take mean of weights
            # You can change this weight aggregation function to aggregate weights differently
            weight = self.agg_func(self.class_weights[cls])
            weights.append(weight)
        return weights

    def calculate_probabilities(self):
        """
        Calculate and store the sampling probabilities based on the weights.

        Returns:
            list: A list of sampling probabilities corresponding to each label.
        """
        total_weight = sum(self.weights)
        probabilities = [w / total_weight for w in self.weights]
        return probabilities

    def __getitem__(self, index):
        """
        Return transformed label information based on the sampled index.
        """
        # Don't use for validation
        if not self.train_mode:
            return self.transforms(self.get_image_and_label(index))
        else:
            index = np.random.choice(len(self.labels), p=self.probabilities)
            return self.transforms(self.get_image_and_label(index))

detection_build.YOLODataset = WeightedInstanceDataset

In [3]:
data_dir = "E:/JordanP/Click-a-Coral/data/reduced/MDBC_Transects_Coral_Sponges"
dataset = f"{data_dir}/YOLO_Detection_Dataset/data.yaml"
project = f"{data_dir}/results"

In [ ]:
args = dict(
    model="E:\\JordanP\\Click-a-Coral\\data\\reduced\\Season_3\\results\\yolov8n\\weights\\best.pt",
    data=dataset,
    project=project,
    name="NL_MutiClass_Corrective_Pretrained_Weighted",
    task='detect',
    epochs=100,
    patience=10,
    half=True,
    imgsz=640,
    single_cls=False,
    plots=True,
    batch=32,
    workers=0,
    save_period=1,
)

trainer = CustomTrainer(overrides=args)
trainer.train()

Ultralytics 8.3.101  Python-3.10.16 torch-2.6.0+cu118 CUDA:0 (Tesla T4, 16384MiB)
engine\trainer: task=detect, mode=train, model=E:\JordanP\Click-a-Coral\data\reduced\Season_1\results\yolov8n\weights\best.pt, data=E:/JordanP/Click-a-Coral/data/reduced/MDBC_Transects_Coral_Sponges/YOLO_Detection_Dataset/data.yaml, epochs=100, time=None, patience=10, batch=32, imgsz=640, save=True, save_period=1, cache=False, device=None, workers=0, project=E:/JordanP/Click-a-Coral/data/reduced/MDBC_Transects_Coral_Sponges/results, name=NL_MutiClass_Corrective_Pretrained_Weighted, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=True, dnn=False, plots=True, source=None, vid_stride=

train: Scanning E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\YOLO_Detection_Dataset\train\labels.cache... 7087 images, 0 backgrounds, 0 corrupt: 100%|██████████| 7087/7087 [00:00<?, ?it/s]

train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\YOLO_Detection_Dataset\train\images\6756696_160219.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\YOLO_Detection_Dataset\train\images\6761625_114983.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\YOLO_Detection_Dataset\train\images\6761625_122537.jpg: 3 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\YOLO_Detection_Dataset\train\images\6761625_122546.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\YOLO_Detection_Dataset\train\images\6761625_122560.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\YOLO_Detection_Dataset\train\images\6761625_122565.jpg: 1 duplicate labels removed
trai


val: Scanning E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\YOLO_Detection_Dataset\valid\labels.cache... 886 images, 0 backgrounds, 0 corrupt: 100%|██████████| 886/886 [00:00<?, ?it/s]

val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\YOLO_Detection_Dataset\valid\images\6761625_122549.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\YOLO_Detection_Dataset\valid\images\6761625_122584.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\YOLO_Detection_Dataset\valid\images\6761625_122587.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\YOLO_Detection_Dataset\valid\images\6761625_122691.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\YOLO_Detection_Dataset\valid\images\6761625_126253.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\YOLO_Detection_Dataset\valid\images\6761625_126456.jpg: 1 duplicate labels removed
val: WARNING  E:

...

     71/100       4.3G      1.368     0.6208      1.253         41        640: 100%|██████████| 222/222 [04:09<00:00,  1.12s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 14/14 [00:20<00:00,  1.44s/it]

                   all        886       1475      0.304      0.385      0.249      0.118
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 61, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.


Calculating new threshold based on 48269 False Positives.

71 epochs completed in 5.330 hours.
Optimizer stripped from E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\results\NL_MutiClass_Corrective_Pretrained_Weighted\weights\last.pt, 6.2MB
Optimizer stripped from E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\results\NL_MutiClass_Corrective_Pretrained_Weighted\weights\best.pt, 6.2MB

Validating E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects_Coral_Sponges\results\NL_MutiClass_Corrective_Pretrained_Weighted\weights\best.pt...
Ultralytics 8.3.101  Python-3.10.16 torch-2.6.0+cu118 CUDA:0 (Tesla T4, 16384MiB)
Model summary (fused): 72 layers, 3,007,793 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 14/14 [00:22<00:00,  1.64s/it]


                   all        886       1475      0.276       0.45      0.271      0.134
  ANTIPATHES_ATLANTICA        134        170      0.293      0.518      0.309      0.147
    ANTIPATHES_FURCATA        142        177      0.345      0.531      0.348      0.152
               BEBRYCE          9         11      0.198      0.562      0.185     0.0647
          CRINOID_STAR        153        194      0.307      0.211       0.18      0.074
              MADRACIS          6          8     0.0691       0.25      0.163      0.101
             MADREPORA         54         70      0.222      0.543      0.238     0.0971
       MURICEA_PENDULA         39         47      0.325      0.617      0.395      0.244
                SPONGE        164        241      0.302      0.158      0.142     0.0595
        SWIFTIA_EXERTA         41         64      0.284      0.562      0.329      0.227
          THESEA_NIVEA         93        141      0.316       0.61      0.382      0.171
                  WHI